# CRIVO — Fase 6b v5: o 46M com a fase 5 em primeiro plano e escolhido pelo sentido (L4, A100 ou T4)

Parte da base crescida da fase 6a (46M, 24 camadas, contexto de 1024 tokens, conferida pelo SHA do `resumo_fase6a.json`) e ensina o CRIVO a conversar. **Ninguém escreve as falas dele:** os alvos de treino são respostas escritas por pessoas e textos do acervo (`experimentos/fase6b_conversa_humana/gerar_fase6b.py`):

1. **OASST2 em português** (Apache-2.0): conversas inteiras escritas por voluntários; ficam fora as mensagens sintéticas, apagadas ou reprovadas na revisão;
2. **legendas pt-BR** (OpenSubtitles v2018, OPUS; uso não comercial, crédito a opensubtitles.org): falas seguidas de filmes e séries;
3. **acervo do CRIVO**: a pergunta "me fala sobre X" e, como resposta, os fatos do próprio acervo (ensina a usar o que recebe no contexto).

**O que a v5 muda da v4** (v3: sentido 62,1% no blind v3 contra 63,5% da v4 F; 9/15 nas frases reais contra 14/15; a melhor versão foi escolhida pela perda, no passo 900, quando a fase 5 só tinha passado pela metade):
- **mais fase 5:** os ~247 mil exemplos entram 3 vezes; as legendas, 1 vez (eram 2); o OASST2, 4;
- **identidade sem conflito:** as situações da fase 5 que o IDENTIDADE.md cobre (quem é, nome, ser humano, robô, sentir) saem do treino, porque duas respostas diferentes para a mesma pergunta o confundiam. A régua aceita as respostas do IDENTIDADE.md nessas situações, e o resultado mostra as duas contas: com a lista da v4 F e com o IDENTIDADE.md;
- **escolha pelo sentido:** a cada medida, 400 frases do dev da fase 5 (frases novas); fica a versão com mais sentido certo;
- **prova também sem o freio de repetição** no blind v3 e nas frases reais, do mesmo jeito que a v4 F foi medida.

**O que a v3 acrescentou à v2** (`experimentos/fase6b_conversa_humana/RESULTADO.md`: na v2 a identidade e as lembranças funcionaram, mas cumprimento, risada e conversa comum não; ele punha a frase de identidade em qualquer lugar):
- **as conversas da fase 5 v4** (`gerar_v4.py`, ~247 mil exemplos, as mesmas que formaram a v4 F do chat): cumprimentos, reações, "não sei", nomes, urgência, fatos do acervo;
- **a prova da v4 F:** o mesmo sentido (`sentido.py`) nos blinds v3, v1, paráfrases, Grok e v4 e nas 15 frases reais, gerados com o mesmo contexto que o chat monta. Assim dá para comparar direto com a F.

**O que a v2 tinha mudado da primeira rodada** (`experimentos/fase6b_conversa_humana/RESULTADO.md`: respostas em laço, jeito de site de "dicas", sem usar `quem` nem `lembrancas`):
- **identidade e lembrança** (`extras_fase6b.py`): perguntas como "quem é você?" com respostas tiradas do `IDENTIDADE.md`, e perguntas sobre o que a pessoa contou ("qual é o meu nome?") com a resposta dentro das lembranças, ou "Você ainda não me contou isso.";
- **muito mais conversa:** todas as ~370 mil legendas 2 vezes, o OASST2 6 vezes (sem as respostas em que o assistente diz ser outro sistema), o acervo 2 vezes;
- **escolha da versão** pela média da perda nas legendas e no OASST2 separados, não só no OASST2 (que a base já tinha lido);
- **freio de repetição na fala** (penalidade nos tokens já ditos e sem repetir trechos de 3 tokens), como vai ser no chat.

O contexto traz `quem` (o resumo do `IDENTIDADE.md`: ele lê quem é e se apresenta do jeito dele) e `lembrancas` (o que a pessoa disse antes, para ele aprender a lembrar). Um quarto de cada lote continua sendo leitura (web, corpus próprio e o `IDENTIDADE.md` inteiro), para ele não esquecer o que leu.

**Quanto treina:** o número de passos sai do tamanho real dos dados (impresso na célula 3), até 2.500 passos. Na A100, uns 15 a 25 minutos.

**Medidas:** perda nas conversas separadas (antes e depois), perda de leitura (para ver se esqueceu), e respostas dele, sem retoque, às frases reais do criador, a "quem é você?", a perguntas com fatos e a testes de lembrança.

> **GPU → Executar tudo.** A100 é a mais rápida; se não houver, L4 serve (uns 3 a 4 vezes mais lenta) e T4 também (mais lenta ainda). O notebook ajusta o lote à memória da GPU, e o resultado é o mesmo. Precisa da fase 6a terminada na mesma conta (`CRIVO/transformer-crivo-46m/resumo_fase6a.json`). Pode parar e continuar: "Executar tudo" de novo continua do checkpoint. Saída: `CRIVO/transformer-crivo-46m-conversa-v4/`; o resumo pequeno é `resumo_fase6b.json`.

## Experimento v5 — reforço de situações difíceis

A v4 ficou em torno de 65–66% de acerto de sentido no blind v3. Os diagnósticos anteriores apontaram dificuldades em cansaço, insultos, pedidos de brevidade, entradas sem sentido e trabalho ruim. A v5 repete **somente exemplos de treino** dessas categorias, sem criar respostas novas. Os antigos blinds passam a ser diagnóstico exploratório (não prova independente), porque as categorias orientaram este experimento. A escolha do checkpoint continua usando `sentido_dev`; não promover pesos sem uma nova bateria independente.


In [ ]:
# Célula 0 — GPU, Drive e a base crescida da fase 6a
import os, sys, json, shutil, subprocess, hashlib, time, math, random, glob, re, gzip
from pathlib import Path
from datetime import datetime
from collections import Counter
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np
import torch
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → A100, L4 ou T4.")
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| PyTorch", torch.__version__, flush=True)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # T4 e mais antigas: float16 (bfloat16 lá é emulado, lento e sem atenção eficiente)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-crivo-46m" / "resumo_fase6a.json").is_file()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei CRIVO/transformer-crivo-46m/resumo_fase6a.json. Rode na conta onde a fase 6a treinou (hr87676620).")
R6A = json.loads((CRIVO / "transformer-crivo-46m" / "resumo_fase6a.json").read_text())
if not R6A.get("terminou") or not R6A.get("pesos_sha256"):
    raise SystemExit("ERRO: a fase 6a ainda não terminou. Termine a 6a primeiro ('Executar tudo' no notebook dela).")
BASE = Path(R6A["pasta"]) / "pesos.pt"
if not BASE.is_file():   # o Drive pode estar montado em outro caminho
    BASE = CRIVO / "transformer-crivo-46m" / Path(R6A["pasta"]).name / "pesos.pt"
if sha256(BASE) != R6A["pesos_sha256"]:
    raise SystemExit(f"ERRO: {BASE} não tem o SHA do resumo da 6a ({R6A['pesos_sha256'][:8]}…).")
BASE_SHA = R6A["pesos_sha256"]
SAIDA = CRIVO / "transformer-crivo-46m-conversa-v4"
SAIDA.mkdir(parents=True, exist_ok=True)
DADOS_DRIVE = CRIVO / "dados-fase6b"
DADOS_DRIVE.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/fase6b")
LOCAL.mkdir(parents=True, exist_ok=True)
dispositivo = torch.device("cuda")
SEED = 20261015
print("CRIVO:", CRIVO, "| base 6a:", BASE, BASE_SHA[:12], flush=True)


In [ ]:
# Célula 1 — código do CRIVO
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
for p in (str(REPO), str(REPO / "scripts"), str(REPO / "experimentos" / "fase6b_conversa_humana"),
          str(REPO / "experimentos" / "fase5_conversa_20261013")):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import formato_fase6, gerar_fase6b, extras_fase6b, cenas_v4
import baixar_fontes_linguagem
import importlib.util
_spec = importlib.util.spec_from_file_location("formato_fase5", REPO / "artefatos" / "conversa_v4f_pt" / "formato_fase5.py")
formato_fase5 = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(formato_fase5)
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(str(BASE.parent / "tokenizer.json"))
ESP = {t: TOK.token_to_id(t) for t in ("<pad>", "<documento>", "<usuario>", "<assistente>", "<fim>")}
DOC, FIM = ESP["<documento>"], ESP["<fim>"]
QUEM = gerar_fase6b.resumo_identidade()
# Fase 5: dados v1–v3 do repositório (SHA conferido) e v4 gerada aqui, como no notebook da v4.
EXP5 = REPO / "experimentos" / "fase5_conversa_20261013"
M = {n: json.loads((EXP5 / f).read_text(encoding="utf-8")) for n, f in (("v1", "manifesto.json"), ("v2", "manifesto_v2.json"),
                                                                       ("v3", "manifesto_v3.json"), ("v4", "manifesto_v4.json"))}
for arq, esperado in (("dados_fase5.json.gz", M["v1"]["arquivo_gz_sha256"]), ("dados_fase5_v2.json.gz", M["v2"]["arquivo_gz_sha256"]),
                      ("dados_fase5_v3.json.gz", M["v3"]["arquivo_gz_sha256"]), ("sentido.py", M["v4"]["sentido_sha256"]),
                      ("cenas_v4.py", M["v4"]["cenas_v4_sha256"]), ("gerar_v4.py", M["v4"]["gerar_v4_sha256"])):
    if sha256(EXP5 / arq) != esperado:
        raise SystemExit(f"ERRO: {arq} difere do manifesto")
V4_SHA_REGISTRADO = M["v4"]["dados_sha256"]
subprocess.run([sys.executable, str(EXP5 / "gerar_v4.py")], cwd=EXP5, check=True)
V4_SHA = json.loads((EXP5 / "manifesto_v4.json").read_text(encoding="utf-8"))["dados_sha256"]
print("dados da fase 5 v4:", "conferidos" if V4_SHA == V4_SHA_REGISTRADO else f"AVISO: sha {V4_SHA[:12]} ≠ registrado {V4_SHA_REGISTRADO[:12]}", flush=True)
import sentido
sentido_ok = cenas_v4.registrar_no_sentido(sentido)
ACEITAS_IDENT = extras_fase6b.aceitas_na_fase5(sentido.norm)   # respostas do IDENTIDADE.md, certas nessas situações
F5 = {n: json.loads(gzip.open(EXP5 / f).read().decode("utf-8")) for n, f in (("v1", "dados_fase5.json.gz"), ("v2", "dados_fase5_v2.json.gz"),
                                                                             ("v3", "dados_fase5_v3.json.gz"), ("v4", "dados_fase5_v4.json.gz"))}
F_SENTIDO = {k: v["sentido"]["certo"] for k, v in json.loads((EXP5 / "resultado_v4_resumo.json").read_text())["variantes"]["F"]["testes"].items()
             if isinstance(v, dict) and "sentido" in v}
print("commit:", COMMIT, "| quem:", QUEM, "| v4 F (chat) no sentido:", F_SENTIDO, flush=True)


In [ ]:
# Célula 2 — dados de conversa (uma vez; depois vêm do Drive) e tokens
DADOS = DADOS_DRIVE / "dados_fase6b.json.gz"
if not DADOS.is_file():
    OASST = LOCAL / "oasst2.messages.jsonl.gz"
    if not OASST.is_file():
        baixar_fontes_linguagem.baixar("oasst2", LOCAL)
    LEGENDAS = gerar_fase6b.LEGENDAS_URL
    subprocess.run([sys.executable, str(REPO / "experimentos/fase6b_conversa_humana/gerar_fase6b.py"), "--oasst", str(OASST),
                    "--legendas", LEGENDAS, "--saida", str(LOCAL / "dados")], cwd=REPO, check=True)
    shutil.copyfile(LOCAL / "dados" / "manifesto_fase6b.json", DADOS_DRIVE / "manifesto_fase6b.json")
    shutil.copyfile(LOCAL / "dados" / "dados_fase6b.json.gz", DADOS)
MANIFESTO_DADOS = json.loads((DADOS_DRIVE / "manifesto_fase6b.json").read_text())
with gzip.open(DADOS, "rb") as f:
    bruto = f.read()
assert hashlib.sha256(bruto).hexdigest() == MANIFESTO_DADOS["dados_sha256"], "dados_fase6b.json.gz corrompido"
DADOS_CONVERSA = json.loads(bruto); del bruto
REL_EXTRAS = extras_fase6b.extras(DADOS_CONVERSA, QUEM)   # identidade, lembranças e filtro do OASST2 (determinístico)

def de_fase5(c):
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(c["historico"])]
    return {"origem": "fase5", "ctx": {"quem": QUEM, **(c["contexto"] or {})},
            "turnos": t + [{"papel": "usuario", "texto": c["mensagem"]}, {"papel": "assistente", "texto": c["resposta"]}]}
FASE5_TREINO = [c for c in F5["v4"]["treino"] if c["situacao"][5:] not in extras_fase6b.FASE5_SUBSTITUIDAS]
DADOS_CONVERSA["treino"] += [de_fase5(c) for c in FASE5_TREINO]

# Reforço v5: repete 1 vez (2x no total, além das repetições globais) apenas exemplos
# do conjunto de TREINO em categorias que a avaliação anterior mostrou como frágeis.
# Não adiciona respostas escritas aqui: reaproveita os alvos originais e não toca no dev/teste.
PESO_REFORCO_SITUACAO = {"cansado": 1, "insulto": 1, "seja_breve": 1, "sem_sentido": 1, "trabalho_ruim": 1}
FASE5_REFORCADOS = [c for c in FASE5_TREINO if c["situacao"][5:] in PESO_REFORCO_SITUACAO]
for c in FASE5_REFORCADOS:
    DADOS_CONVERSA["treino"].extend([de_fase5(c)] * PESO_REFORCO_SITUACAO[c["situacao"][5:]])
print("reforço v5 por situação:", dict(Counter(c["situacao"][5:] for c in FASE5_REFORCADOS)), flush=True)

DADOS_CONVERSA["val"] += [de_fase5(c) for c in F5["v4"]["dev"]]
REL_EXTRAS["fase5_treino"], REL_EXTRAS["fase5_dev"] = len(F5["v4"]["treino"]), len(F5["v4"]["dev"])
print("acrescentados:", REL_EXTRAS, flush=True)
print("dados:", json.dumps(MANIFESTO_DADOS["por_origem"], ensure_ascii=False), flush=True)
print("relatório:", json.dumps(MANIFESTO_DADOS["relatorio"], ensure_ascii=False), flush=True)

# tokens: cada texto distinto é codificado uma vez, em lote
_textos = set()
for v in DADOS_CONVERSA.values():
    for e in v:
        _textos.add(formato_fase6.limpar(formato_fase6.serializar_contexto(e.get("ctx"))))
        _textos.update(formato_fase6.limpar(t["texto"]) for t in e["turnos"])
_textos = sorted(_textos)
CACHE = dict(zip(_textos, (e.ids for e in TOK.encode_batch(_textos, add_special_tokens=False))))
del _textos
def codificar(texto):
    ids = CACHE.get(texto)
    return ids if ids is not None else TOK.encode(texto, add_special_tokens=False).ids

CTX = 1024
EX = {}   # (parte, origem) -> lista de (ids uint16, alvo bool)
nao_cabe = Counter()
for parte, v in DADOS_CONVERSA.items():
    for e in v:
        try:
            ids, alvo = formato_fase6.montar(codificar, ESP, e.get("ctx"), e["turnos"], contexto=CTX)
        except ValueError:
            nao_cabe[e["origem"]] += 1
            continue
        EX.setdefault((parte, e["origem"]), []).append((np.asarray(ids, np.uint16), np.asarray(alvo, bool)))
TOKENS_ALVO = {f"{p}/{o}": int(sum(a.sum() for _, a in l)) for (p, o), l in sorted(EX.items())}
print("exemplos:", {f"{p}/{o}": len(l) for (p, o), l in sorted(EX.items())}, "| não couberam:", dict(nao_cabe), flush=True)
print("tokens com perda:", TOKENS_ALVO, flush=True)

# leitura: corpus próprio, uma parte do FineWeb-2 (da 6a) e o IDENTIDADE.md
for n in ("treino.bin", "validacao.bin"):
    if not (LOCAL / n).is_file():
        shutil.copyfile(CRIVO / "dados-17m" / n, LOCAL / n)
PROPRIO = np.memmap(LOCAL / "treino.bin", dtype=np.uint16, mode="r")
PROPRIO_VAL = np.memmap(LOCAL / "validacao.bin", dtype=np.uint16, mode="r")
MAN_WEB = json.loads((CRIVO / "corpus-fineweb2-pt" / "manifesto.json").read_text())
for p in (MAN_WEB["partes"][-1], MAN_WEB["val"]):   # a última parte: a que a 6a leu por último
    if not (LOCAL / p["arquivo"]).is_file() or (LOCAL / p["arquivo"]).stat().st_size != 2 * p["tokens"]:
        shutil.copyfile(CRIVO / "corpus-fineweb2-pt" / p["arquivo"], LOCAL / p["arquivo"])
WEB = np.memmap(LOCAL / MAN_WEB["partes"][-1]["arquivo"], dtype=np.uint16, mode="r")
WEB_VAL = np.memmap(LOCAL / MAN_WEB["val"]["arquivo"], dtype=np.uint16, mode="r")
IDENT = np.asarray([DOC] + TOK.encode(formato_fase6.limpar((REPO / "IDENTIDADE.md").read_text(encoding="utf-8")),
                                      add_special_tokens=False).ids + [FIM], np.uint16)
print(f"leitura: próprio {len(PROPRIO)/1e9:.2f} bi | web {len(WEB)/1e6:.0f}M | IDENTIDADE.md {len(IDENT)} tokens", flush=True)


In [ ]:
# Célula 3 — o plano de treino (determinístico: recomeçar dá os mesmos lotes)
REPETIR = {"fase5": 3, "oasst2": 4, "acervo": 2, "legendas": 1, "identidade": 4, "lembranca": 1}   # vezes que cada exemplo entra
MAX_PASSOS = 3400
LOTE, CONVERSA_POR_LOTE = 64, 48
L = CTX + 1

def empacotar(exemplos, rng):
    """Junta conversas em sequências de CTX+1 tokens; o resto da sequência é <fim> sem perda."""
    ordem = rng.permutation(len(exemplos))
    seqs, alvos, ids_cur, alvo_cur = [], [], [], []
    def fechar():
        ids = np.concatenate(ids_cur); alvo = np.concatenate(alvo_cur)
        s = np.full(L, FIM, np.uint16); a = np.zeros(L, bool)
        s[:len(ids)] = ids; a[:len(alvo)] = alvo
        seqs.append(s); alvos.append(a)
    n = 0
    for i in ordem:
        ids, alvo = exemplos[i]
        if n + len(ids) > L and ids_cur:
            fechar(); ids_cur, alvo_cur, n = [], [], 0
        ids_cur.append(ids); alvo_cur.append(alvo); n += len(ids)
    if ids_cur:
        fechar()
    return seqs, alvos

def amostra_por_tokens(lista, tokens, rng, inicio=0):
    """Pega exemplos de `lista` (já embaralhada) a partir de `inicio` até somar `tokens`."""
    out, soma, i = [], 0, inicio
    while soma < tokens and i < len(lista):
        out.append(lista[i]); soma += len(lista[i][0]); i += 1
    return out, i

rng_plano = np.random.default_rng(SEED)
tokens_de = lambda lista: sum(len(i) for i, _ in lista)
por_origem = {o: EX.get(("treino", o), []) for o in REPETIR}
legendas = por_origem["legendas"]
legendas = [legendas[i] for i in rng_plano.permutation(len(legendas))]
resto = sum(REPETIR[o] * tokens_de(l) for o, l in por_origem.items() if o != "legendas")
cabe = MAX_PASSOS * CONVERSA_POR_LOTE * L * 0.92 - resto     # ~92% de cada sequência é ocupada
if tokens_de(legendas) > cabe:                                # passou do teto: entram menos legendas
    legendas = legendas[:max(1, int(len(legendas) * cabe / tokens_de(legendas)))]
por_origem["legendas"] = legendas
todos = [e for o, l in por_origem.items() for _ in range(REPETIR[o]) for e in l]
s_, a_ = empacotar(todos, np.random.default_rng(SEED + 1))
n = len(s_) - len(s_) % CONVERSA_POR_LOTE
SEQ, ALVO = np.stack(s_[:n]), np.stack(a_[:n]); del s_, a_, todos
PASSOS = len(SEQ) // CONVERSA_POR_LOTE
PLANO = {o: {"exemplos": len(l), "vezes": REPETIR[o], "tokens": tokens_de(l) * REPETIR[o]} for o, l in por_origem.items()}
LEGENDAS_INEDITAS = len(legendas) < len(EX.get(("treino", "legendas"), []))
print("plano:", json.dumps(PLANO, ensure_ascii=False), flush=True)
print(f"PASSOS: {PASSOS} ({len(SEQ)} sequências de conversa + {LOTE - CONVERSA_POR_LOTE} de leitura por passo) | "
      f"{100 * ALVO.mean():.0f}% dos tokens de conversa têm perda", flush=True)
if PASSOS < 50:
    raise SystemExit("ERRO: poucos dados de conversa. Mande o print para o assistente.")

def val_pacotes(parte, origem, max_ex=600):
    lista = EX.get((parte, origem), [])[:max_ex]
    if not lista:
        return None
    s, a = empacotar(lista, np.random.default_rng(SEED))
    return np.stack(s), np.stack(a)
VAL = {f"{p}/{o}": val_pacotes(p, o) for p in ("val", "teste") for o in REPETIR}
VAL = {k: v for k, v in VAL.items() if v is not None}
SELECAO = ["sentido_dev (400 frases novas do dev da fase 5, com o IDENTIDADE.md aceito)"]
nota = lambda m: -m["sentido_dev"]   # menor é melhor no laço de treino: o sentido entra com sinal trocado
print("medidas separadas:", {k: len(v[0]) for k, v in VAL.items()}, "| escolha da melhor versão por", SELECAO, flush=True)


In [ ]:
# Célula 4 — o CRIVO da 6a, ou o checkpoint
CKPT = SAIDA / "checkpoint.pt"
salvo = torch.load(BASE, map_location="cpu", weights_only=True)
CFG = dict(salvo["config"], dropout=0.0)
assert CFG["contexto"] >= CTX, CFG
modelo = lp.LinguagemProfunda(lp.Configuracao(**CFG)).to(dispositivo)
modelo.load_state_dict(salvo["modelo"]); del salvo
print(f"base 6a: {CFG['camadas']} camadas, {sum(p.numel() for p in modelo.parameters())/1e6:.1f}M parâmetros, contexto {CFG['contexto']}", flush=True)
PASSO0, HIST, MELHOR_PERDA, MELHOR_ESTADO, MELHOR_PASSO = 0, [], float("inf"), None, 0
if CKPT.is_file():
    c = torch.load(CKPT, map_location="cpu", weights_only=False)
    if c.get("base_sha256") != BASE_SHA or c.get("passos") != PASSOS:
        raise SystemExit("ERRO: o checkpoint é de outra base ou de outro plano de dados. Apague checkpoint.pt para recomeçar.")
    modelo.load_state_dict(c["modelo"]); PASSO0, HIST, MELHOR_PERDA = c["passo"], c["historico"], c["melhor_perda"]
    MELHOR_ESTADO, MELHOR_PASSO = c["melhor_modelo"], c["melhor_passo"]
    print("continuando do checkpoint, passo", PASSO0, flush=True)


In [ ]:
# Célula 5 — treinar: 48 sequências de conversa + 16 de leitura por passo
LR, LR_MIN, AQUECIMENTO = 1e-4, 1e-5, 100
PESO_LEITURA = 0.5
MEDIR_A_CADA = max(50, PASSOS // 9)
MEMORIA_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
MICRO = 1 if MEMORIA_GB > 30 else 2 if MEMORIA_GB > 20 else 8   # A100 inteiro; L4 em 2; T4 em 8
print(f"GPU com {MEMORIA_GB:.0f} GB: lote em {MICRO} pedaço(s)", flush=True)
SALVAR_A_CADA, MAX_HORAS = 500, 11.5   # cada checkpoint substituído vai para a lixeira do Drive (e ocupa espaço)

params = list(modelo.parameters())
otim = torch.optim.AdamW([{"params": [p for p in params if p.dim() >= 2], "weight_decay": 0.1},
                          {"params": [p for p in params if p.dim() < 2], "weight_decay": 0.0}], lr=LR, betas=(0.9, 0.95), fused=True)
if CKPT.is_file():
    otim.load_state_dict(c["otim"]); del c
escala = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
try:
    rodar = torch.compile(modelo)
except Exception as erro:
    print("sem torch.compile:", erro, flush=True); rodar = modelo
taxa = lambda p: LR * (p + 1) / AQUECIMENTO if p < AQUECIMENTO else LR_MIN + (LR - LR_MIN) * 0.5 * (1 + math.cos(math.pi * min(1.0, (p - AQUECIMENTO) / max(1, PASSOS - AQUECIMENTO))))

def alvos_de(seq, alvo):
    """x = seq[:-1]; y = seq[1:] onde o token tem perda, -100 no resto."""
    s = torch.from_numpy(seq.astype(np.int64))
    y = s[:, 1:].clone(); y[~torch.from_numpy(alvo[:, 1:])] = -100
    return s[:, :-1], y

def leitura(rng, n):
    out = []
    for _ in range(n):
        u = rng.random()
        if u < 0.03:      # o IDENTIDADE.md, inteiro ou um pedaço
            ini = int(rng.integers(0, max(1, len(IDENT) - L)))
            pedaco = IDENT[ini:ini + L]
            s = np.full(L, FIM, np.uint16); s[:len(pedaco)] = pedaco
            a = np.zeros(L, bool); a[1:len(pedaco)] = True
        else:
            fonte = WEB if u < 0.55 else PROPRIO
            i = int(rng.integers(0, len(fonte) - L))
            s = np.asarray(fonte[i:i + L]); a = np.ones(L, bool)
        out.append((s, a))
    return np.stack([s for s, _ in out]), np.stack([a for _, a in out])

def lote(passo):
    i = (passo - 1) * CONVERSA_POR_LOTE
    xc, yc = alvos_de(SEQ[i:i + CONVERSA_POR_LOTE], ALVO[i:i + CONVERSA_POR_LOTE])
    xl, yl = alvos_de(*leitura(np.random.default_rng(SEED * 7 + passo), LOTE - CONVERSA_POR_LOTE))
    return xc.to(dispositivo), yc.to(dispositivo), xl.to(dispositivo), yl.to(dispositivo)

@torch.no_grad()
def perda_pacotes(seq, alvo, tam=None):
    tam = tam or max(2, 16 // MICRO)
    modelo.eval(); soma, n = 0.0, 0
    for j in range(0, len(seq), tam):
        x, y = alvos_de(seq[j:j + tam], alvo[j:j + tam])
        x, y = x.to(dispositivo), y.to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(x)[0]
        soma += F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), y.reshape(-1), ignore_index=-100, reduction="sum").item()
        n += int((y != -100).sum())
    return soma / max(1, n)

@torch.no_grad()
def perda_leitura(fonte, ctx, lotes=15, tam=None):
    tam = tam or max(2, 16 // MICRO)
    modelo.eval(); g = np.random.default_rng(SEED); r = []
    for _ in range(lotes):
        ini = g.integers(0, len(fonte) - ctx - 1, size=tam)
        b = torch.from_numpy(np.stack([fonte[i:i + ctx + 1] for i in ini]).astype(np.int64)).to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            r.append(modelo(b[:, :-1], b[:, 1:])[1].item())
    return float(np.mean(r))

def medir(passo, separadas=("val",)):
    m = {"passo": passo}
    for k, (s, a) in VAL.items():
        if k.split("/")[0] in separadas:
            m["perda_" + k] = perda_pacotes(s, a)
    m["leitura_proprio_256"] = perda_leitura(PROPRIO_VAL, 256)
    m["leitura_web_1024"] = perda_leitura(WEB_VAL, CTX)
    m["sentido_dev"] = certo_ident(avaliar_blind(DEV_SENTIDO))
    print(f"  MEDIDA passo {passo}: " + " | ".join(f"{k} {v:.4f}" for k, v in m.items() if k != "passo"), flush=True)
    return m

def salvar(passo):
    tmp = SAIDA / "checkpoint.tmp"
    torch.save({"modelo": modelo.state_dict(), "otim": otim.state_dict(), "passo": passo, "historico": HIST,
                "melhor_perda": MELHOR_PERDA, "melhor_modelo": MELHOR_ESTADO, "melhor_passo": MELHOR_PASSO, "config": CFG, "commit": COMMIT, "base_sha256": BASE_SHA, "passos": PASSOS}, tmp)
    tmp.replace(CKPT)

PROIBIDOS = [ESP["<pad>"], DOC, ESP["<usuario>"], ESP["<assistente>"]]
FREIO = {"penalidade": 1.3, "sem_repetir_ngrama": 3, "top_k": 40}
SEM_FREIO = {"penalidade": 1.0, "sem_repetir_ngrama": 10 ** 9, "top_k": 40}   # como a v4 F foi medida

def caso_ids(c):
    ctx = {"quem": QUEM, **(c["contexto"] or {})}
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(c["historico"])]
    ids, _ = formato_fase6.montar(codificar, ESP, ctx, t + [{"papel": "usuario", "texto": c["mensagem"]}], contexto=CTX - 120, gerar=True)
    return ids

@torch.no_grad()
def gerar_lote_freado(prompts, max_novos=110):
    # Gulosa, várias de uma vez (prompts do mesmo tamanho), com o mesmo freio de repetição da célula 6.
    modelo.eval()
    ids = torch.tensor(prompts, device=dispositivo)
    vistos = torch.zeros(len(prompts), CFG["vocabulario"], dtype=torch.bool, device=dispositivo)
    vivos, saidas, fim = list(range(len(prompts))), [[] for _ in prompts], [False] * len(prompts)
    k = FREIO["sem_repetir_ngrama"] - 1
    for _ in range(max_novos):
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(ids)[0][:, -1].float()
        lg[:, PROIBIDOS] = -float("inf")
        lg = torch.where(vistos, torch.where(lg > 0, lg / FREIO["penalidade"], lg * FREIO["penalidade"]), lg)
        for j, i in enumerate(vivos):
            s_ = saidas[i]
            if len(s_) >= k:
                proib = [s_[a + k] for a in range(len(s_) - k) if s_[a:a + k] == s_[-k:]]
                if proib:
                    lg[j, proib] = -float("inf")
        nxt = lg.argmax(-1).tolist()
        manter = []
        for j, (i, t) in enumerate(zip(vivos, nxt)):
            if t == FIM:
                fim[i] = True
            else:
                saidas[i].append(t); manter.append(j)
        if not manter:
            break
        idx = torch.tensor(manter, device=dispositivo)
        novos = torch.tensor([[nxt[j]] for j in manter], device=dispositivo)
        ids = torch.cat([ids[idx], novos], 1)
        vistos = vistos[idx]; vistos.scatter_(1, novos, True)
        vivos = [vivos[j] for j in manter]
    return saidas, fim

def avaliar_blind(casos, lote_max=None):
    lote_max = lote_max or (96 if MEMORIA_GB > 30 else 32)   # GPU menor: menos frases por vez
    prep = []
    for c in casos:
        try:
            prep.append((c, caso_ids(c)))
        except ValueError:
            pass
    grupos = {}
    for n, (_, p) in enumerate(prep):
        grupos.setdefault(len(p), []).append(n)
    textos, fins = [None] * len(prep), [False] * len(prep)
    for _, idx in sorted(grupos.items()):
        for a in range(0, len(idx), lote_max):
            parte = idx[a:a + lote_max]
            sa, fi = gerar_lote_freado([prep[n][1] for n in parte])
            for n, s_, f_ in zip(parte, sa, fi):
                textos[n], fins[n] = TOK.decode(s_).strip(), f_
    linhas = []
    for (c, _), texto, fim_ in zip(prep, textos, fins):
        av = formato_fase5.avaliar_saida(c, texto, fim_)
        extra = (c.get("exigir") or {}).get("contem_texto")
        if extra:
            av["contem_texto"] = extra.casefold() in texto.casefold()
            av["tudo_ok"] = av["tudo_ok"] and av["contem_texto"]
        linha = {"situacao": c["situacao"], "contraexemplo": bool(c.get("contraexemplo")), "mensagem": c["mensagem"],
                 "alvo": c["resposta"], "saida": texto, **av}
        if c.get("aceita"):
            linha["sentido"] = cenas_v4.julgar_real(sentido, c, texto)
            linha["aceita"], linha["historico"] = c["aceita"], c["historico"][-2:]
        else:
            j = sentido.julgar(linha)
            linha["sentido"] = None if j is None else j[0]
            if j is not None and j[0] != "certo" and sentido.norm(texto) in ACEITAS_IDENT.get(c["situacao"][5:], ()):
                linha["sentido_ident"] = "certo"   # resposta do IDENTIDADE.md nessa situação
        linhas.append(linha)
    return linhas

def certo_ident(linhas):
    # Sentido certo contando também as respostas do IDENTIDADE.md nas situações que ele cobre.
    cenas = [l for l in linhas if l["sentido"] is not None]
    return round(sum(l["sentido"] == "certo" or l.get("sentido_ident") == "certo" for l in cenas) / max(1, len(cenas)), 4)

def estratificar(casos, n, semente):
    por = {}
    for c in casos:
        por.setdefault(c["situacao"], []).append(c)
    rr = random.Random(semente)
    for v in por.values():
        rr.shuffle(v)
    out = []
    while len(out) < n and any(por.values()):
        for k_ in sorted(por):
            if por[k_] and len(out) < n:
                out.append(por[k_].pop())
    return out

DEV_SENTIDO = estratificar([c for c in F5["v4"]["dev"] if c.get("frase_nova") and c["situacao"].startswith("cena:")], 400, SEED + 3)

if PASSO0 == 0:
    HIST.append(medir(0, ("val", "teste")))   # a base da 6a, antes de conversar
t0, passo = time.time(), PASSO0
for passo in range(PASSO0 + 1, PASSOS + 1):
    modelo.train()
    for g in otim.param_groups:
        g["lr"] = taxa(passo)
    xc, yc, xl, yl = lote(passo)
    otim.zero_grad(set_to_none=True)
    soma_conv = soma_leit = 0.0
    for parte in range(MICRO):   # GPU menor: o mesmo lote em pedaços, gradiente acumulado
        fc = slice(parte * len(xc) // MICRO, (parte + 1) * len(xc) // MICRO)
        fl = slice(parte * len(xl) // MICRO, (parte + 1) * len(xl) // MICRO)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = rodar(torch.cat([xc[fc], xl[fl]]))[0].float()
        V, nc = lg.shape[-1], fc.stop - fc.start
        l_conv = F.cross_entropy(lg[:nc].reshape(-1, V), yc[fc].reshape(-1), ignore_index=-100)
        l_leit = F.cross_entropy(lg[nc:].reshape(-1, V), yl[fl].reshape(-1), ignore_index=-100) if len(lg) > nc else lg.sum() * 0
        escala.scale((l_conv + PESO_LEITURA * l_leit) / MICRO).backward()
        soma_conv += l_conv.item() / MICRO; soma_leit += l_leit.item() / MICRO
        del lg
    escala.unscale_(otim)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    escala.step(otim); escala.update()
    if passo % 50 == 0:
        print(f"passo {passo}/{PASSOS} conversa {soma_conv:.3f} leitura {soma_leit:.3f} lr {taxa(passo):.2e} | "
              f"{(time.time()-t0)/max(1,passo-PASSO0):.2f}s/passo", flush=True)
    if passo % MEDIR_A_CADA == 0 or passo == PASSOS:
        HIST.append(medir(passo))
        if nota(HIST[-1]) < MELHOR_PERDA:
            MELHOR_PERDA = nota(HIST[-1])
            MELHOR_ESTADO = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}   # na memória, não no Drive
            MELHOR_PASSO = passo
            print(f"  ↑ melhor até agora (sentido no dev {-MELHOR_PERDA:.3f})", flush=True)
    fim_tempo = time.time() - t0 > MAX_HORAS * 3600
    if passo % SALVAR_A_CADA == 0 or passo == PASSOS or fim_tempo:
        salvar(passo)
        print(f"  checkpoint salvo (passo {passo})", flush=True)
    if fim_tempo:
        print("Limite de tempo da sessão: rode 'Executar tudo' de novo para continuar.", flush=True)
        break
TERMINOU = passo >= PASSOS


In [ ]:
# Célula 6 — a melhor versão: medidas finais e o que ele diz (sem retoque)
if not TERMINOU:
    raise SystemExit("Ainda não terminou: rode 'Executar tudo' de novo para continuar.")
modelo.load_state_dict(MELHOR_ESTADO); PASSO_MELHOR = MELHOR_PASSO
FINAL = medir(PASSO_MELHOR, ("val", "teste"))


@torch.no_grad()
def gerar_freado(ids, max_tokens, temperatura, semente):
    # Como modelo.gerar, com freio de repetição: tokens já ditos perdem força e nenhum trecho de 3 tokens se repete.
    modelo.eval(); g = torch.Generator(device=dispositivo).manual_seed(semente)
    ids, saida = list(ids), []
    for _ in range(max_tokens):
        x = torch.tensor([ids[-CFG["contexto"]:]], dtype=torch.long, device=dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(x)[0][0, -1].float()
        lg[PROIBIDOS] = -float("inf")
        if saida:
            vistos = torch.tensor(sorted(set(saida)), device=dispositivo)
            v = lg[vistos]; lg[vistos] = torch.where(v > 0, v / FREIO["penalidade"], v * FREIO["penalidade"])
        k = FREIO["sem_repetir_ngrama"] - 1
        if len(saida) >= k:
            proib = [saida[i + k] for i in range(len(saida) - k) if saida[i:i + k] == saida[-k:]]
            if proib:
                lg[proib] = -float("inf")
        if temperatura == 0:
            prox = int(lg.argmax())
        else:
            lg = lg / temperatura
            lg[lg < lg.topk(FREIO["top_k"]).values[-1]] = -float("inf")
            prox = int(torch.multinomial(lg.softmax(-1), 1, generator=g))
        if prox == FIM:
            return saida, True
        saida.append(prox); ids.append(prox)
    return saida, False

def falar(ctx, turnos, temperatura=0.0, semente=0, max_tokens=120):
    ids, _ = formato_fase6.montar(codificar, ESP, ctx, turnos, contexto=CTX - max_tokens, gerar=True)
    saida, terminou = gerar_freado(ids, max_tokens, temperatura, semente)
    return TOK.decode(saida).strip(), terminou

def turnos_de(historico, mensagem):
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(historico)]
    return t + [{"papel": "usuario", "texto": mensagem}]

def conferir(ctx, historico, mensagem, texto, terminou, exigir=None):
    caso = {"contexto": {k: v for k, v in (ctx or {}).items() if k in formato_fase5.CAMPOS}, "historico": list(historico),
            "mensagem": mensagem + " " + " ".join((ctx or {}).get("lembrancas", [])) + " " + (ctx or {}).get("quem", ""),
            "exigir": exigir or {}}
    r = formato_fase5.avaliar_saida(caso, texto, terminou)
    return {k: v for k, v in r.items() if k in ("tudo_ok", "usa_fato", "inventados", "sem_repeticao", "terminou")}

FALAS = []
def ouvir(grupo, ctx, historico, mensagem, exigir=None, espera=None):
    for temp, sem in ((0.0, 0), (0.7, 1)):
        texto, terminou = falar(ctx, turnos_de(historico, mensagem), temp, sem)
        FALAS.append({"grupo": grupo, "temperatura": temp, "historico": list(historico), "mensagem": mensagem,
                      "lembrancas": (ctx or {}).get("lembrancas"), "fatos": (ctx or {}).get("fatos"), "resposta": texto,
                      "terminou": terminou, "conferencia": conferir(ctx, historico, mensagem, texto, terminou, exigir),
                      "lembrou": (espera.casefold() in texto.casefold()) if espera else None})

Q = {"quem": QUEM}
for c in cenas_v4.CONVERSAS_REAIS:
    ouvir("frases_reais_do_criador", Q, c["historico"], c["mensagem"])
for m in ("quem é você?", "qual é o seu nome?", "você é uma pessoa?", "o que você mais gosta de fazer?", "você tem sentimentos?"):
    ouvir("identidade_treinada", Q, [], m)        # formas que estão no treino
for m in extras_fase6b.PERGUNTAS_AVALIACAO + ["do que você é feito?"]:
    ouvir("identidade_nova", Q, [], m)            # formas que não estão no treino
for m in ("me dá uma dica pra estudar melhor", "como faço arroz?", "por que o céu é azul?", "o que é mais importante na vida?",
          "tô cansado do trabalho", "me conta uma curiosidade", "qual a diferença entre vírus e bactéria?", "você gosta de música?"):
    ouvir("perguntas_livres", Q, [], m)
# Valores, tipos e perguntas fora do treino (extras_fase6b.FATOS não tem papagaio, irmã, Juiz de Fora, Remo, filosofia...)
for lembr, msg, espera in ((["a pessoa disse: meu nome é Rosângela"], "você sabe como eu me chamo?", "Rosângela"),
                           (["a pessoa disse: tenho um papagaio chamado Louro"], "qual o nome do meu papagaio?", "Louro"),
                           (["a pessoa disse: eu moro em Juiz de Fora", "a pessoa disse: tenho 40 anos"], "em que cidade eu vivo?", "Juiz de Fora"),
                           (["a pessoa disse: semana que vem tenho prova de filosofia"], "minha prova é de quê?", "filosofia"),
                           (["a pessoa disse: eu torço pro Remo"], "lembra qual é o meu time?", "Remo"),
                           (["a pessoa disse: minha irmã se chama Lúcia", "a pessoa disse: ela tem 7 anos"], "quantos anos tem a minha irmã?", "7"),
                           (["a pessoa disse: meu nome é Tânia"], "qual é o meu time?", "não me contou"),
                           (["a pessoa disse: eu moro em Natal"], "qual é o nome do meu cachorro?", "não me contou")):
    ouvir("lembrancas", {"quem": QUEM, "lembrancas": lembr}, [], msg, espera=espera)
for e in DADOS_CONVERSA.get("teste", []):
    if e["origem"] == "acervo" and sum(f["grupo"] == "fatos_do_acervo" for f in FALAS) < 40:
        ouvir("fatos_do_acervo", e["ctx"], [], e["turnos"][0]["texto"], exigir={"fato": True})
for e in DADOS_CONVERSA.get("teste", []):
    if e["origem"] == "oasst2" and sum(f["grupo"] == "oasst2_separado" for f in FALAS) < 30             and e["turnos"][-2]["texto"] not in {f["mensagem"] for f in FALAS}:
        ts = e["turnos"][:-1]
        ouvir("oasst2_separado", e["ctx"], [t["texto"] for t in ts[:-1]], ts[-1]["texto"])

def taxa_de(grupo, chave):
    v = [f["conferencia"].get(chave) if chave != "lembrou" else f["lembrou"] for f in FALAS if f["grupo"] == grupo and f["temperatura"] == 0.0]
    v = [x for x in v if x is not None]
    return round(sum(v) / len(v), 3) if v else None
TAXAS = {"conferencia_tudo_ok": {g: taxa_de(g, "tudo_ok") for g in sorted({f["grupo"] for f in FALAS})},
         "usa_fato_acervo": taxa_de("fatos_do_acervo", "usa_fato"), "lembrou": taxa_de("lembrancas", "lembrou")}
for f in FALAS:
    if f["temperatura"] == 0.0 and f["grupo"] in ("frases_reais_do_criador", "identidade_treinada", "identidade_nova", "lembrancas", "perguntas_livres"):
        print(f"[{f['grupo']}] {f['mensagem']!r}\n   → {f['resposta']}", flush=True)
print(json.dumps(TAXAS, ensure_ascii=False, indent=1), flush=True)


In [ ]:
# Célula 6b — diagnóstico v5. Os blinds antigos são exploratórios porque as categorias orientaram o reforço; não são holdout independente.
BLINDS = {"v3": F5["v3"]["blind_v3"], "v1": F5["v1"]["blind"], "parafrases": F5["v2"]["blind_parafrase"],
          "grok": F5["v2"]["blind_grok"], "v4": F5["v4"]["blind_v4"]}
SENTIDO = {}
for k, casos in BLINDS.items():
    t0 = time.time()
    ls = avaliar_blind(casos)
    a = sentido.analisar(ls, k)
    SENTIDO[k] = {"certo": a["certo"], "certo_com_identidade": certo_ident(ls), "confundiu": a["confundiu"], "n": a["n_cenas"], "tudo_ok": formato_fase5.resumir(ls).get("tudo_ok"),
                  "piores": a["piores"][:6],
                  "exemplos": [{x: l[x] for x in ("situacao", "mensagem", "saida", "sentido")} for l in ls[::max(1, len(ls) // 12)]][:12]}
    f_ = F_SENTIDO.get(k)
    print(f"sentido {k}: {100 * (a['certo'] or 0):.1f}% (lista da F) | {100 * SENTIDO[k]['certo_com_identidade']:.1f}% (com o IDENTIDADE.md)"
          + (f" | v4 F, no chat: {100 * f_:.1f}%" if f_ is not None else "")
          + f" | {time.time() - t0:.0f}s", flush=True)
FREIO_ATUAL = FREIO
FREIO = SEM_FREIO
ls_sem = avaliar_blind(F5["v3"]["blind_v3"])
SEM_FREIO_V3 = {"certo": sentido.analisar(ls_sem, "v3")["certo"], "certo_com_identidade": certo_ident(ls_sem)}
ls_sem = avaliar_blind(F5["v4"]["conversas_reais"])
SEM_FREIO_REAIS = sum(l["sentido"] == "certo" for l in ls_sem)
FREIO = FREIO_ATUAL
print(f"sem freio: sentido v3 {100 * (SEM_FREIO_V3['certo'] or 0):.1f}% (com o IDENTIDADE.md {100 * SEM_FREIO_V3['certo_com_identidade']:.1f}%) | "
      f"frases reais {SEM_FREIO_REAIS}/15", flush=True)
ls = avaliar_blind(F5["v4"]["conversas_reais"])
REAIS = {"certo": sum(l["sentido"] == "certo" for l in ls), "n": len(ls),
         "linhas": [{x: l.get(x) for x in ("historico", "mensagem", "saida", "sentido", "aceita")} for l in ls]}
print(f"\nfrases reais do criador: {REAIS['certo']}/{REAIS['n']} certas (v4 F: 14/15)", flush=True)
for l in REAIS["linhas"]:
    print(f"  [{l['sentido']}] {l['mensagem']!r}\n     → {l['saida']}", flush=True)


In [ ]:
# Célula 7 — salvar a versão que conversa e o resumo pequeno
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST = SAIDA / f"conversa_{stamp}"
DEST.mkdir(parents=True)
torch.save({"versao": lp.VERSAO, "modelo": {k: v.detach().cpu() for k, v in modelo.state_dict().items()}, "config": CFG,
            "passo": PASSO_MELHOR, "execucao": {"tokenizer_sha256": sha256(BASE.parent / "tokenizer.json"), "pesos_externos": False,
                                                 "base_6a_sha256": BASE_SHA, "protocolo": "fase6b", "commit": COMMIT,
                                                 "dados_sha256": MANIFESTO_DADOS["dados_sha256"]}}, DEST / "pesos.pt")
shutil.copyfile(BASE.parent / "tokenizer.json", DEST / "tokenizer.json")
shutil.copyfile(REPO / "experimentos/fase6b_conversa_humana/formato_fase6.py", DEST / "formato_fase6.py")
resumo = {"protocolo": "fase6b_v4", "criado_em": datetime.now().isoformat(), "commit": COMMIT, "gpu": GPU,
          "base_6a_sha256": BASE_SHA, "config": CFG, "passos": PASSOS, "passo_escolhido": PASSO_MELHOR, "selecao": SELECAO,
          "plano": PLANO, "extras": REL_EXTRAS, "freio": FREIO, "sentido": SENTIDO, "frases_reais": REAIS, "sem_freio": {"v3": SEM_FREIO_V3, "frases_reais": SEM_FREIO_REAIS}, "f_sentido": F_SENTIDO, "dados_fase5_v4_sha256": V4_SHA, "legendas_acabaram": LEGENDAS_INEDITAS, "dados": MANIFESTO_DADOS,
          "tokens_com_perda": TOKENS_ALVO, "nao_couberam": dict(nao_cabe), "antes": HIST[0], "final": FINAL, "historico": HIST,
          "taxas": TAXAS, "falas": FALAS, "pasta": str(DEST), "pesos_sha256": sha256(DEST / "pesos.pt"),
          "falas_escritas_pelo_assistente": "fase 5 (listas da v4), lembranças (frases curtas) e identidade (do IDENTIDADE.md)", "ativo_no_chat": False}
(SAIDA / "resumo_fase6b.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=1, default=float))
print("SALVO EM", DEST, "| sha", resumo["pesos_sha256"], flush=True)
print("Avise o assistente: ele lê o resumo_fase6b.json.", flush=True)
